# ETF Portfolio Backtester - Demo Notebook

**DADS 4002 Course Project**

This notebook demonstrates how to use the ETF Portfolio Backtester system interactively.

---

## 📦 Step 1: Import Libraries and Modules

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Import project modules with correct aliases
from modules.db_connector import DatabaseConnector, DatabaseConfig
from modules.backtest_engine import MomentumBacktester as BacktestEngine
from modules.analytics import PortfolioAnalytics as Analytics
from modules.crud_operations import CRUDOperations

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

%matplotlib inline

print("✓ All modules imported successfully!")

## 🔌 Step 2: Connect to Database

In [ ]:
# Configure database connection
# IMPORTANT: Update these credentials to match your MySQL setup
config = DatabaseConfig()
config.host = '127.0.0.1'
config.port = 3306
config.user = 'root'
config.password = 'krittanut123456'  # Update this to your MySQL password
config.database = 'etf_backtester_db'

# Initialize database connection
db = DatabaseConnector(config)

# Test connection
if db.test_connection():
    print("✓ Database connected successfully!\n")
    
    # Check data
    etf_count = db.get_table_count('ETF_Master')
    price_count = db.get_table_count('Price_Data')
    
    print(f"✓ ETF_Master: {etf_count} ETFs")
    print(f"✓ Price_Data: {price_count:,} records")
else:
    print("✗ Cannot connect to database")
    print("Please check MySQL connection settings")

## 📊 Step 3: View ETF Master Data

In [ ]:
# Query all ETFs
query = """
SELECT
    ETF_ID,
    Ticker_Symbol,
    ETF_Name,
    Asset_Type,
    Expense_Ratio
FROM ETF_Master
ORDER BY Asset_Type, Ticker_Symbol
"""

results = db.execute_query_dict(query)
df_etfs = pd.DataFrame(results)

print(f"Total ETFs: {len(df_etfs)}\n")

# Display first 10
print("Sample ETFs:")
print(df_etfs.head(10))

# Count by Asset Type
print("\nETF Count by Asset Type:")
print(df_etfs.groupby('Asset_Type').size())

## 📈 Step 4: View Recent Price Data (SPY Example)

In [ ]:
# Query SPY recent prices
query = """
SELECT
    pd.Price_Date,
    pd.Open_Price,
    pd.High_Price,
    pd.Low_Price,
    pd.Close_Price,
    pd.Volume,
    em.Ticker_Symbol
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE em.Ticker_Symbol = 'SPY'
ORDER BY pd.Price_Date DESC
LIMIT 10
"""

results = db.execute_query_dict(query)
df_spy = pd.DataFrame(results)

print("SPY - Latest 10 Weeks:")
print(df_spy)

# Plot
df_spy_sorted = df_spy.sort_values('Price_Date')
plt.figure(figsize=(12, 6))
plt.plot(df_spy_sorted['Price_Date'], df_spy_sorted['Close_Price'], marker='o', linewidth=2)
plt.title('SPY - Close Price (Last 10 Weeks)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 🎯 Step 5: Run Backtest

In [ ]:
# Initialize backtest engine
backtest = BacktestEngine(db)

# Define date range (last 6 months)
end_date = datetime.now()
start_date = end_date - timedelta(days=180)

# Set parameters
lookback_days = 90
top_n = 5
holding_period_days = 30
rebalance_days = 30

print("Running Backtest...")
print("=" * 60)
print(f"Start Date: {start_date.date()}")
print(f"End Date: {end_date.date()}")
print(f"Lookback Period: {lookback_days} days")
print(f"Top N ETFs: {top_n}")
print(f"Holding Period: {holding_period_days} days")
print(f"Rebalance Every: {rebalance_days} days")
print("=" * 60)

# Run backtest
results = backtest.run_backtest(
    start_date=start_date.strftime('%Y-%m-%d'),
    end_date=end_date.strftime('%Y-%m-%d'),
    lookback_days=lookback_days,
    holding_period_days=holding_period_days,
    rebalance_days=rebalance_days,
    top_n=top_n
)

# Display results
if results:
    print("\n✓ Backtest completed successfully!")
    print(f"Backtest Run ID: {results['run_id']}")
    print(f"\n{'='*60}")
    print(f"Total Rebalances: {results['total_rebalances']}")
    print(f"Cumulative Return: {results['cumulative_return']*100:.2f}%")
    print(f"Avg Return per Period: {results['avg_return_per_period']*100:.2f}%")
    print(f"Approximate CAGR: {results['cagr']*100:.2f}%")
    print(f"{'='*60}")
else:
    print("✗ Backtest failed")

## 📊 Step 6: Analytics - Volatility Analysis

In [ ]:
# Initialize analytics
analytics = Analytics(db)

print("\n" + "="*60)
print("ANALYTICS: Volatility Analysis by Asset Type")
print("="*60)

# Get volatility data (uses correct method name)
volatility_results = analytics.analyze_volatility_by_asset_type()

if volatility_results:
    df_volatility = pd.DataFrame(volatility_results)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(df_volatility['Asset_Type'], df_volatility['Annualized_Volatility_Pct'], 
            color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
    plt.title('Annualized Volatility by Asset Type', fontsize=14, fontweight='bold')
    plt.xlabel('Asset Type')
    plt.ylabel('Annualized Volatility (%)')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Volatility analysis complete!")
else:
    print("\n✗ No volatility data available")

## 📊 Step 7: Analytics - Lookback Period Optimization

In [ ]:
print("\n" + "="*60)
print("ANALYTICS: Lookback Period Optimization")
print("="*60)

# Compare different lookback periods (90 days vs 180 days)
lookback_results = analytics.compare_lookback_periods([90, 180])

if lookback_results:
    df_lookback = pd.DataFrame(lookback_results)
    
    # Plot if we have numeric CAGR data
    if 'CAGR_Pct' in df_lookback.columns and not df_lookback['CAGR_Pct'].isna().all():
        plt.figure(figsize=(10, 6))
        plt.plot(df_lookback['Lookback_Period_Days'], df_lookback['CAGR_Pct'], 
                 marker='o', linewidth=2, markersize=8, color='#2ca02c')
        plt.title('CAGR by Lookback Period', fontsize=14, fontweight='bold')
        plt.xlabel('Lookback Period (Days)')
        plt.ylabel('CAGR (%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    print("\n✓ Lookback period analysis complete!")
else:
    print("\n✗ No lookback comparison data available")
    print("   (This requires multiple backtest runs with different lookback periods)")

## 📊 Step 8: Analytics - Drawdown Analysis

In [ ]:
print("\n" + "="*60)
print("ANALYTICS: Drawdown Exposure Analysis")
print("="*60)

# Analyze which asset types were held during maximum drawdowns
drawdown_results = analytics.analyze_drawdown_exposure()

if drawdown_results:
    df_drawdown = pd.DataFrame(drawdown_results)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(df_drawdown['Asset_Type'], df_drawdown['Num_Occurrences'],
            color=['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c'])
    plt.title('Asset Holdings During Drawdown Periods', fontsize=14, fontweight='bold')
    plt.xlabel('Asset Type')
    plt.ylabel('Number of Occurrences')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Drawdown analysis complete!")
else:
    print("\n✗ No drawdown data available")

## 🔍 Step 9: CRUD Operations - View Latest Backtest

In [ ]:
# Initialize CRUD operations
crud = CRUDOperations(db)

print("\n" + "="*60)
print("CRUD: View Latest Backtest Results")
print("="*60)

# Read latest backtest results (uses correct method name)
results = crud.read_backtest_results(limit=50)

if results:
    # Convert to DataFrame for analysis
    df_results = pd.DataFrame(results)
    
    # Show summary statistics
    print("\n📊 Summary Statistics:")
    print(f"  Total Selections: {len(df_results)}")
    print(f"  Unique Dates: {df_results['Selection_Date'].nunique()}")
    print(f"  Average Momentum Score: {df_results['Momentum_Score'].mean():.2f}%")
    
    if df_results['Holding_Return'].notna().any():
        avg_return = df_results['Holding_Return'].mean() * 100
        print(f"  Average Holding Return: {avg_return:.2f}%")
    
    print("\n✓ Latest backtest displayed successfully!")
else:
    print("\n✗ No backtest results found")

## 📈 Step 10: Advanced Analysis - Cumulative Returns

In [ ]:
# Get 1 year of data for top 5 ETFs
query = """
SELECT
    em.Ticker_Symbol,
    em.Asset_Type,
    pd.Price_Date,
    pd.Close_Price
FROM Price_Data pd
JOIN ETF_Master em ON pd.ETF_ID = em.ETF_ID
WHERE pd.Price_Date >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
  AND em.Ticker_Symbol IN ('SPY', 'QQQ', 'AGG', 'GLD', 'TLT')
ORDER BY em.Ticker_Symbol, pd.Price_Date
"""

results = db.execute_query_dict(query)
df = pd.DataFrame(results)

if not df.empty:
    # Pivot table
    pivot = df.pivot_table(
        index='Price_Date',
        columns='Ticker_Symbol',
        values='Close_Price'
    )
    
    # Calculate returns
    returns = pivot.pct_change()
    
    # Calculate cumulative returns
    cumulative_returns = (1 + returns).cumprod()
    
    # Plot
    plt.figure(figsize=(14, 7))
    
    for ticker in cumulative_returns.columns:
        plt.plot(cumulative_returns.index, cumulative_returns[ticker], 
                label=ticker, linewidth=2)
    
    plt.title('Cumulative Returns - Last 1 Year', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nSummary Statistics (Annualized):")
    annual_returns = returns.mean() * 52
    annual_vol = returns.std() * (52 ** 0.5)
    
    summary = pd.DataFrame({
        'Annual Return': annual_returns,
        'Annual Volatility': annual_vol,
        'Sharpe Ratio': annual_returns / annual_vol
    })
    
    print(summary)
else:
    print("No data available")

## 🔒 Step 11: Close Database Connection

In [ ]:
# Close database connection
db.close_pool()
print("✓ Database connection closed successfully")
print("\nThank you for using ETF Portfolio Backtester!")

---

## 📚 Notes

- **All data is REAL from Yahoo Finance** (not synthetic)
- **50 real ETFs** across 4 asset types
- **10 years** of weekly price data
- **~25,933 price records** in total

## ✅ Verified Working (20 Nov 2025)

This notebook has been tested and verified to work 100% with:
- Fixed import statements with correct aliases
- Correct method names for all operations
- Proper database configuration
- Updated backtest parameters

---

**DADS 4002 Course Project** | Made with ❤️